# N154 · SQL子查询、CTE与反连接

**目标：** 把复杂查询拆成可验证中间关系。

**先修：** N153。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** EXISTS/NOT EXISTS；相关子查询；CTE；UNION/UNION ALL；去重；关系除法扩展。

## 从问题到算法

反连接描述“找不到关联记录”。NOT IN遇到子查询中的NULL会产生UNKNOWN，因此NOT EXISTS更直接表达逐行不存在。部门最高薪要保留全部并列者，不是每组随便取一行。买齐所有产品可用双重NOT EXISTS表达“没有任何缺失产品”，自然处理重复购买。

## 最小实现

**本章接口：** `query_customers_without_orders.sql`、`query_department_max.sql`、`query_bought_all.sql`

In [1]:
import sqlite3

SQL_QUERIES = {
    'query_customers_without_orders.sql': r'''SELECT c.name AS Customers
FROM Customers AS c
WHERE NOT EXISTS (SELECT 1 FROM Orders AS o WHERE o.customerId = c.id);''',
    'query_department_max.sql': r'''WITH maximum AS (
    SELECT departmentId, MAX(salary) AS highest
    FROM Employee GROUP BY departmentId
)
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee AS e
JOIN maximum AS m ON m.departmentId = e.departmentId AND e.salary = m.highest
JOIN Department AS d ON d.id = e.departmentId;''',
    'query_bought_all.sql': r'''SELECT DISTINCT c.customer_id
FROM Customer AS c
WHERE NOT EXISTS (
    SELECT 1 FROM Product AS p
    WHERE NOT EXISTS (
        SELECT 1 FROM Customer AS bought
        WHERE bought.customer_id = c.customer_id AND bought.product_key = p.product_key
    )
);''',
}

def create_sqlite_fixture():
    connection=sqlite3.connect(':memory:')
    connection.executescript(r'''
CREATE TABLE Customers(id INTEGER PRIMARY KEY,name TEXT);
CREATE TABLE Orders(id INTEGER PRIMARY KEY,customerId INTEGER);
INSERT INTO Customers VALUES (1,'A'),(2,'B'),(3,'C');
INSERT INTO Orders VALUES (1,1),(2,NULL),(3,1);
CREATE TABLE Department(id INTEGER PRIMARY KEY,name TEXT);
CREATE TABLE Employee(id INTEGER PRIMARY KEY,name TEXT,salary INTEGER,departmentId INTEGER);
INSERT INTO Department VALUES (1,'R&D'),(2,'Sales');
INSERT INTO Employee VALUES (1,'Ada',100,1),(2,'Ben',100,1),(3,'Cal',90,1),(4,'Dee',80,2);
CREATE TABLE Product(product_key INTEGER PRIMARY KEY);
CREATE TABLE Customer(customer_id INTEGER,product_key INTEGER);
INSERT INTO Product VALUES (1),(2);
INSERT INTO Customer VALUES (1,1),(1,1),(1,2),(2,1),(2,1),(3,2);
''')
    return connection

def query_rows(connection,name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

## 正确性、前提与复杂度

NOT EXISTS只测试是否存在至少一条匹配，不受无关NULL和重复订单影响。聚合最高薪再按部门与工资双键连接，恰保留该部门所有达到最大值的行。双重否定等价于对每个产品都存在购买记录；产品表为空时条件真，返回购买表中可识别的所有客户ID，不凭空创造客户全集。

**代价：** 相关子查询在无索引时可能重复扫描，数据库可优化为连接。教学重点是逻辑语义正确；需要规模优化时为关联键建立索引并查看计划，而不是以CTE语法本身声称更快。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

db=create_sqlite_fixture()
show_table(['反连接方式','实际结果'],[('NOT EXISTS',query_rows(db,'query_customers_without_orders.sql')),('含NULL的NOT IN',db.execute('SELECT name FROM Customers WHERE id NOT IN (SELECT customerId FROM Orders)').fetchall())])
for name in ['query_department_max.sql','query_bought_all.sql']:
    cursor=db.execute(SQL_QUERIES[name]); print(name); show_table([x[0] for x in cursor.description],cursor.fetchall())
db.close()

反连接方式,实际结果
NOT EXISTS,"[('B',), ('C',)]"
含NULL的NOT IN,[]


query_department_max.sql


Department,Employee,Salary
R&D,Ada,100
R&D,Ben,100
Sales,Dee,80


query_bought_all.sql


customer_id
1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
db=create_sqlite_fixture()
assert set(query_rows(db,'query_customers_without_orders.sql'))=={('B',),('C',)}
assert set(query_rows(db,'query_department_max.sql'))=={('R&D','Ada',100),('R&D','Ben',100),('Sales','Dee',80)}
assert query_rows(db,'query_bought_all.sql')==[(1,)]
db.execute('DELETE FROM Orders'); assert len(query_rows(db,'query_customers_without_orders.sql'))==3
db.execute('DELETE FROM Product'); assert set(query_rows(db,'query_bought_all.sql'))=={(1,),(2,),(3,)}
db.close()
print('N154: 所有本章断言通过')

N154: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用NOT EXISTS避免NOT IN与NULL陷阱。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较最高薪并列处理。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 183. Customers Who Never Order（canonical）
- 184. Department Highest Salary（canonical）
- 1045. Customers Who Bought All Products（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。